In [1]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
print("Project root added to sys.path")

Project root added to sys.path


In [2]:
import torch
import torch.nn 
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
import os

from clients.federated_training import federated_training
from utils.data_partition import dirichlet_partition


torch.manual_seed(42)
np.random.seed(42)

In [3]:
import importlib.util
import sys
import os

# Path to the actual Python file
module_path = os.path.abspath('../datasets/speech_commands_dataset.py')

# Module name and loader
spec = importlib.util.spec_from_file_location("speech_commands_dataset", module_path)
speech_commands_dataset = importlib.util.module_from_spec(spec)
spec.loader.exec_module(speech_commands_dataset)

# Use the class
SpeechCommandsDataset = speech_commands_dataset.SpeechCommandsDataset



In [4]:
# Create dataset instances
train_dataset = SpeechCommandsDataset(root='./data', split='train', fixed_length=32)
test_dataset = SpeechCommandsDataset(root='./data', split='test', fixed_length=32)


print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

# Get a sample
sample_data, sample_label = train_dataset[0]
print(f"\nSample data shape: {sample_data.shape}")
print(f"Sample label: {sample_label}")
print(f"Label class: {train_dataset.classes[sample_label]}")


Train dataset size: 84843
Test dataset size: 11005

Sample data shape: torch.Size([40, 32])
Sample label: 0
Label class: backward


In [5]:
num_clients = 5
partitions = {
    "iid":   dirichlet_partition(train_dataset, num_clients=num_clients, alpha=0.01),
}

c:\Users\la7tim\Desktop\Internship\FedTinyProp\utils\data_partition.py:5: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  labels = np.array(dataset.targets)


In [6]:
from models.config import get_tinyprop_config
config = get_tinyprop_config("speechcommands")
print("\nModel Configuration:")
for key, value in config.items():
    print(f"{key}: {value}")

# Initialize model parameters
tinyprop_params = config["tinyprop_params"]
print("\nTinyProp Parameters:")
print(f"S_min: {tinyprop_params.S_min}")
print(f"S_max: {tinyprop_params.S_max}")
print(f"zeta: {tinyprop_params.zeta}")
print(f"number_of_layers: {tinyprop_params.number_of_layers}")


Model Configuration:
tinyprop_params: <models.tinyProp.TinyPropParams object at 0x00000262D95A58D0>
optimizer: {'type': 'sgd', 'lr': 0.005, 'momentum': 0.9, 'weight_decay': 0.0005}
lr_scheduler: {'type': 'cosine', 'T_max': 100, 'eta_min': 0.0001, 'warmup_epochs': 5, 'warmup_start_lr': 0.0001}
gradient_clip: 1.0
batch_size: 32
num_epochs: 1
label_smoothing: 0.1
quantization: {'bits': 8, 'enabled': True, 'adaptive': True, 'min_bits': 4, 'max_bits': 16, 'layer_specific': True, 'error_threshold': 0.01, 'momentum': 0.9}
mfcc_config: {'sample_rate': 16000, 'n_mfcc': 40, 'n_fft': 1024, 'hop_length': 512}

TinyProp Parameters:
S_min: 0.5
S_max: 0.95
zeta: 0.5
number_of_layers: 6


In [8]:
from clients.aggregators import sparse_fedavg_aggregate
def train_and_analyze_partition(partition_name, client_datasets, tinyprop_params):
    print(f"\nTraining on partition: {partition_name.upper()}")
    
    # Get base config
    config = get_tinyprop_config('speechcommands')
    
    # Run training
    model, metrics = federated_training(
        client_datasets=client_datasets,
        model_name='speechcommands',
        testset=test_dataset,
        tinyprop_params=tinyprop_params,
        aggregator_fn=sparse_fedavg_aggregate,
        rounds=100,
        device="cuda" if torch.cuda.is_available() else "cpu",
        local_epochs=1,
        early_stopping_patience=100,
        early_stopping_delta=0.001,
        csv_log_path=f'results/speechcommands_metrics.csv',  # Single CSV file for all partitions
        initial_sparsity=tinyprop_params.S_min,
        target_sparsity=tinyprop_params.S_max,
        energy_budget=1000,
        save_dir='results',
        save_interval=1,
        use_dense_baseline=True
    )
    
    # Print summary statistics
    print(f"\nTraining Summary for {partition_name.upper()}:")
    print(f"Final Accuracy: {metrics['accuracy'][-1]:.4f}")
    print(f"Average Sparsity: {np.mean(metrics['sparsity']):.4f}")
    print(f"Average Compression Ratio: {np.mean(metrics['compression_ratio']):.4f}")
    print(f"Average Effective Compute Ratio: {np.mean(metrics['effective_compute_ratio']):.4f}")
    print(f"Total Communication Cost: {metrics['communication'][-1]/1024:.2f} KB")
    print(f"Total Memory Saved: {metrics['memory_saved'][-1]:.2f} MB")
    
    return model, metrics

# Run training for each partition
partition_results = {}
for strategy_name, client_datasets in partitions.items():
    print(f"\nStarting training for {strategy_name.upper()} partition...")
    try:
        model, metrics = train_and_analyze_partition(strategy_name, client_datasets, tinyprop_params)
        partition_results[strategy_name] = {
            'model': model,
            'metrics': metrics
        }
        print(f"Completed training for {strategy_name.upper()}")
    except Exception as e:
        print(f"Error in {strategy_name.upper()}: {str(e)}")
        continue


Starting training for IID partition...

Training on partition: IID

[Training Debug] Initializing clients...
[Training Debug] Client 0 initialized with 25499 samples
[Training Debug] Client 1 initialized with 24481 samples
[Training Debug] Client 2 initialized with 8497 samples
[Training Debug] Client 3 initialized with 10770 samples
[Training Debug] Client 4 initialized with 15596 samples

Round 1/100

[Training Debug] Training client 0

[Client Debug] Starting training for 1 epochs with batch size 32
[Debug] Round 0 loss threshold: 0.1000

[Training Debug] Training client 1

[Client Debug] Starting training for 1 epochs with batch size 32
[Debug] Round 0 loss threshold: 0.1000

[Training Debug] Training client 2

[Client Debug] Starting training for 1 epochs with batch size 32
[Debug] Round 0 loss threshold: 0.1000

[Training Debug] Training client 3

[Client Debug] Starting training for 1 epochs with batch size 32
[Debug] Round 0 loss threshold: 0.1000

[Training Debug] Training cl